In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

I0000 00:00:1788179277.039903   33427 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788179277.123684   33427 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788179279.522326   33427 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv('student_performance_prediction_dataset-2.csv')

In [3]:
df.head()

,student_id,age,gender,study_hours,attendance,sleep_hours,previous_grade,assignments_completed,practice_tests_taken,group_study_hours,...,social_media_hours,family_income,parent_education,internet_access,device_type,school_type,extracurriculars,final_grade,grade_category,pass_fail
0,1,21,Male,1.645404,79.154521,8.230886,96.053840,7.719620,1.871170,1.447894,...,4.018412,Medium,Master,Yes,Mobile,Private,Coding Club,59.248749,D,Pass
1,2,18,Male,4.462126,72.526685,6.139219,53.024821,6.754758,5.630071,1.891288,...,3.268642,Medium,Master,Yes,Laptop,Public,NaN,58.595595,D,Pass
2,3,19,Female,6.220212,98.531716,6.946313,78.775422,10.000000,7.862877,1.774356,...,2.327293,Low,High School,Yes,Tablet,Private,Music,85.855289,A,Pass
3,4,21,Female,1.826644,97.731245,8.297048,76.122618,7.440486,2.316252,1.204271,...,1.163367,Medium,Bachelor,Yes,Laptop,Public,Debate,42.117503,F,Fail
4,5,17,Male,3.789322,78.589107,6.777171,81.305681,9.962609,5.335697,1.399230,...,0.411183,High,High School,Yes,Laptop,Private,Debate,62.870474,C,Pass


In [4]:
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 25 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   student_id             300000 non-null  int64  
 1   age                    300000 non-null  int64  
 2   gender                 300000 non-null  object 
 3   study_hours            300000 non-null  float64
 4   attendance             300000 non-null  float64
 5   sleep_hours            300000 non-null  float64
 6   previous_grade         300000 non-null  float64
 7   assignments_completed  300000 non-null  float64
 8   practice_tests_taken   300000 non-null  float64
 9   group_study_hours      300000 non-null  float64
 10  notes_quality_score    300000 non-null  float64
 11  time_management_score  300000 non-null  float64
 12  motivation_level       300000 non-null  float64
 13  mental_health_score    300000 non-null  float64
 14  screen_time            300000 non-nu

In [5]:
print("Missing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:")
print(df.duplicated().sum())

Missing Values:
student_id                   0
age                          0
gender                       0
study_hours                  0
attendance                   0
sleep_hours                  0
previous_grade               0
assignments_completed        0
practice_tests_taken         0
group_study_hours            0
notes_quality_score          0
time_management_score        0
motivation_level             0
mental_health_score          0
screen_time                  0
social_media_hours           0
family_income                0
parent_education             0
internet_access              0
device_type              15105
school_type                  0
extracurriculars         49981
final_grade                  0
grade_category               4
pass_fail                    0
dtype: int64

Duplicate Rows:
0


In [6]:
categorical_cols = df.select_dtypes(include=['object']).columns
print("Categorical Columns:", categorical_cols.tolist())
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

Categorical Columns: ['gender', 'family_income', 'parent_education', 'internet_access', 'device_type', 'school_type', 'extracurriculars', 'grade_category', 'pass_fail']

gender:
gender
Female    144083
Male      143871
Other      12046
Name: count, dtype: int64

family_income:
family_income
Medium    134859
Low       105067
High       60074
Name: count, dtype: int64

parent_education:
parent_education
High School    135098
Bachelor       105227
Master          44939
PhD             14736
Name: count, dtype: int64

internet_access:
internet_access
Yes    254951
No      45049
Name: count, dtype: int64

device_type:
device_type
Laptop    164893
Mobile     90116
Tablet     29886
Name: count, dtype: int64

school_type:
school_type
Public     195174
Private    104826
Name: count, dtype: int64

extracurriculars:
extracurriculars
Arts           50235
Coding Club    50187
Music          50097
Sports         49797
Debate         49703
Name: count, dtype: int64

grade_category:
grade_category
F  

In [7]:
df_encoded = df.copy()
for col in categorical_cols:
    df_encoded[col] = pd.Categorical(df_encoded[col]).codes

In [9]:
X = df_encoded.drop('final_grade', axis=1)
y = df_encoded['final_grade']
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (300000, 24)
Target shape: (300000,)


In [ ]:
X_sample = X.sample(n=10000, random_state=42)
y_sample = y[X_sample.index]

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 40000
Testing set size: 10000


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
model1 = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])
model1.compile(optimizer='adam', loss='mse', metrics=['mae'])
history1 = model1.fit(X_train_scaled, y_train, epochs=20, batch_size=64, validation_split=0.2, verbose=0)
y_pred1 = model1.predict(X_test_scaled, verbose=0)
mse1 = mean_squared_error(y_test, y_pred1)
rmse1 = np.sqrt(mse1)
mae1 = mean_absolute_error(y_test, y_pred1)
r2_1 = r2_score(y_test, y_pred1)
print(f"Model 1 (ReLU-Adam-20): MSE={mse1:.4f}, RMSE={rmse1:.4f}, MAE={mae1:.4f}, R²={r2_1:.4f}")

KeyboardInterrupt: 

In [ ]:
model2 = keras.Sequential([
    layers.Dense(100, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(50, activation='tanh'),
    layers.Dense(25, activation='tanh'),
    layers.Dense(1)
])
model2.compile(optimizer='sgd', loss='mse', metrics=['mae'])
history2 = model2.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
y_pred2 = model2.predict(X_test_scaled, verbose=0)
mse2 = mean_squared_error(y_test, y_pred2)
rmse2 = np.sqrt(mse2)
mae2 = mean_absolute_error(y_test, y_pred2)
r2_2 = r2_score(y_test, y_pred2)
print(f"Model 2 (Tanh-SGD-50): MSE={mse2:.4f}, RMSE={rmse2:.4f}, MAE={mae2:.4f}, R²={r2_2:.4f}")

In [ ]:
model3 = keras.Sequential([
    layers.Dense(120, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(60, activation='relu'),
    layers.Dense(30, activation='relu'),
    layers.Dense(1)
])
model3.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history3 = model3.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)
y_pred3 = model3.predict(X_test_scaled, verbose=0)
mse3 = mean_squared_error(y_test, y_pred3)
rmse3 = np.sqrt(mse3)
mae3 = mean_absolute_error(y_test, y_pred3)
r2_3 = r2_score(y_test, y_pred3)
print(f"Model 3 (ReLU-RMSprop-100): MSE={mse3:.4f}, RMSE={rmse3:.4f}, MAE={mae3:.4f}, R²={r2_3:.4f}")

In [ ]:
model4 = keras.Sequential([
    layers.Dense(96, activation='sigmoid', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(48, activation='sigmoid'),
    layers.Dense(24, activation='sigmoid'),
    layers.Dense(1)
])
model4.compile(optimizer='adam', loss='mse', metrics=['mae'])
history4 = model4.fit(X_train_scaled, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
y_pred4 = model4.predict(X_test_scaled, verbose=0)
mse4 = mean_squared_error(y_test, y_pred4)
rmse4 = np.sqrt(mse4)
mae4 = mean_absolute_error(y_test, y_pred4)
r2_4 = r2_score(y_test, y_pred4)
print(f"Model 4 (Sigmoid-Adam-20): MSE={mse4:.4f}, RMSE={rmse4:.4f}, MAE={mae4:.4f}, R²={r2_4:.4f}")

In [ ]:
model5 = keras.Sequential([
    layers.Dense(110, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(55, activation='tanh'),
    layers.Dense(27, activation='tanh'),
    layers.Dense(1)
])
model5.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history5 = model5.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)
y_pred5 = model5.predict(X_test_scaled, verbose=0)
mse5 = mean_squared_error(y_test, y_pred5)
rmse5 = np.sqrt(mse5)
mae5 = mean_absolute_error(y_test, y_pred5)
r2_5 = r2_score(y_test, y_pred5)
print(f"Model 5 (Tanh-RMSprop-100): MSE={mse5:.4f}, RMSE={rmse5:.4f}, MAE={mae5:.4f}, R²={r2_5:.4f}")

In [ ]:
model6 = keras.Sequential([
    layers.Dense(105, activation='sigmoid', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(52, activation='sigmoid'),
    layers.Dense(26, activation='sigmoid'),
    layers.Dense(1)
])
model6.compile(optimizer='sgd', loss='mse', metrics=['mae'])
history6 = model6.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
y_pred6 = model6.predict(X_test_scaled, verbose=0)
mse6 = mean_squared_error(y_test, y_pred6)
rmse6 = np.sqrt(mse6)
mae6 = mean_absolute_error(y_test, y_pred6)
r2_6 = r2_score(y_test, y_pred6)
print(f"Model 6 (Sigmoid-SGD-50): MSE={mse6:.4f}, RMSE={rmse6:.4f}, MAE={mae6:.4f}, R²={r2_6:.4f}")

In [ ]:
comparison_data = {
    'Model': ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Model 6'],
    'Activation': ['ReLU', 'Tanh', 'ReLU', 'Sigmoid', 'Tanh', 'Sigmoid'],
    'Optimizer': ['Adam', 'SGD', 'RMSprop', 'Adam', 'RMSprop', 'SGD'],
    'Epochs': [50, 50, 100, 20, 100, 50],
    'MSE': [mse1, mse2, mse3, mse4, mse5, mse6],
    'RMSE': [rmse1, rmse2, rmse3, rmse4, rmse5, rmse6],
    'MAE': [mae1, mae2, mae3, mae4, mae5, mae6],
    'R² Score': [r2_1, r2_2, r2_3, r2_4, r2_5, r2_6]
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(comparison_df['Model'], comparison_df['MSE'])
axes[0, 0].set_title('MSE Comparison')
axes[0, 0].set_ylabel('MSE')
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].bar(comparison_df['Model'], comparison_df['RMSE'])
axes[0, 1].set_title('RMSE Comparison')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].bar(comparison_df['Model'], comparison_df['MAE'])
axes[1, 0].set_title('MAE Comparison')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].bar(comparison_df['Model'], comparison_df['R² Score'])
axes[1, 1].set_title('R² Score Comparison')
axes[1, 1].set_ylabel('R² Score')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

histories = [history1, history2, history3, history4, history5, history6]
models_names = ['Model 1 (ReLU-Adam-50)', 'Model 2 (Tanh-SGD-50)', 'Model 3 (ReLU-RMSprop-100)', 
                'Model 4 (Sigmoid-Adam-20)', 'Model 5 (Tanh-RMSprop-100)', 'Model 6 (Sigmoid-SGD-50)']

for idx, (history, name) in enumerate(zip(histories, models_names)):
    row = idx // 3
    col = idx % 3
    axes[row, col].plot(history.history['loss'], label='Training Loss')
    axes[row, col].plot(history.history['val_loss'], label='Validation Loss')
    axes[row, col].set_title(name)
    axes[row, col].set_xlabel('Epoch')
    axes[row, col].set_ylabel('Loss')
    axes[row, col].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model_idx = comparison_df['R² Score'].idxmax()
best_model = [model1, model2, model3, model4, model5, model6][best_model_idx]
best_y_pred = [y_pred1, y_pred2, y_pred3, y_pred4, y_pred5, y_pred6][best_model_idx]
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_activation = comparison_df.loc[best_model_idx, 'Activation']
best_optimizer = comparison_df.loc[best_model_idx, 'Optimizer']
best_epochs = comparison_df.loc[best_model_idx, 'Epochs']

print(f"Best Model: {best_model_name}")
print(f"Configuration: {best_activation} - {best_optimizer} - {best_epochs} epochs")
print(f"R² Score: {comparison_df.loc[best_model_idx, 'R² Score']:.4f}")
print(f"\nSample Predictions (First 10 samples):")
print("Actual vs Predicted:")
for i in range(10):
    print(f"Sample {i+1}: Actual = {y_test.iloc[i]:.2f}, Predicted = {best_y_pred[i][0]:.2f}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(y_test, best_y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted - {best_model_name}')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("="*80)
print("FINAL ANALYSIS AND CONCLUSION")
print("="*80)
print("\nModel Performance Summary:")
print(comparison_df.sort_values('R² Score', ascending=False).to_string(index=False))
print(f"\nBest Performing Model: {best_model_name}")
print(f"Activation Function: {best_activation}")
print(f"Optimizer: {best_optimizer}")
print(f"Epochs: {best_epochs}")
print(f"\nPerformance Metrics:")
print(f"MSE: {comparison_df.loc[best_model_idx, 'MSE']:.4f}")
print(f"RMSE: {comparison_df.loc[best_model_idx, 'RMSE']:.4f}")
print(f"MAE: {comparison_df.loc[best_model_idx, 'MAE']:.4f}")
print(f"R² Score: {comparison_df.loc[best_model_idx, 'R² Score']:.4f}")
print("\nConclusion:")
print(f"The {best_activation} activation function with {best_optimizer} optimizer")
print(f"and {best_epochs} epochs provides the best student performance prediction.")
print(f"The model explains {comparison_df.loc[best_model_idx, 'R² Score']*100:.2f}% of the variance in Final Grades.")